In [1]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Audio, display

load_dotenv()

DEEPGRAM_API_KEY = os.getenv("DEEPGRAM_API_KEY")
DEEPGRAM_URL = "https://api.deepgram.com/v1/speak"
# voices under models https://developers.deepgram.com/reference/text-to-speech/speak-request#request.query.model
DEEPGRAM_MODEL = "aura-2-thalia-en"
DEEPGRAM_ENCODING = "mp3"
DEEPGRAM_CONTAINER = None  # "wav", "ogg", or None
DEEPGRAM_SAMPLE_RATE = None  # e.g., 22050


## Deepgram (TTS)

Reference: https://developers.deepgram.com/reference/text-to-speech/speak-request


In [2]:
def deepgram_tts_bytes(text: str, output_path: str | None = "outputs/deepgram.mp3") -> bytes:
    if not DEEPGRAM_API_KEY:
        raise ValueError("Missing DEEPGRAM_API_KEY in notebooks/.env")

    params = {"model": DEEPGRAM_MODEL}
    if DEEPGRAM_ENCODING:
        params["encoding"] = DEEPGRAM_ENCODING
    if DEEPGRAM_CONTAINER:
        params["container"] = DEEPGRAM_CONTAINER
    if DEEPGRAM_SAMPLE_RATE:
        params["sample_rate"] = DEEPGRAM_SAMPLE_RATE

    # Strip whitespace and any accidental surrounding quotes from .env
    auth = DEEPGRAM_API_KEY.strip().strip("'")
    if not (auth.startswith("Token ") or auth.startswith("Bearer ")):
        auth = f"Token {auth}"

    headers = {
        "Authorization": auth,
        "Content-Type": "application/json",
    }
    payload = {"text": text}

    r = requests.post(DEEPGRAM_URL, params=params, json=payload, headers=headers, timeout=60)
    if r.status_code == 401:
        detail = r.text.strip() or "Unauthorized"
        raise RuntimeError(
            "Deepgram 401 Unauthorized. Check DEEPGRAM_API_KEY in notebooks/.env, "
            "restart the kernel after editing, and ensure the key is active. "
            f"Response: {detail}"
        )
    r.raise_for_status()

    audio = r.content
    if output_path:
        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(audio)
    return audio



In [3]:
audio = deepgram_tts_bytes("Hello from Deepgram. This is a quick TTS smoke test.")


In [4]:
len(audio)


23184

In [5]:
display(Audio(audio))
